In [74]:
import argparse
import copy
import gc
import hashlib
import itertools
import logging
import math
import os
import shutil
import warnings
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import torch.utils.checkpoint
import transformers
from accelerate import Accelerator
from accelerate.logging import get_logger
from accelerate.utils import ProjectConfiguration, set_seed
from huggingface_hub import create_repo, upload_folder
from packaging import version
from PIL import Image
from PIL.ImageOps import exif_transpose
from torch.utils.data import Dataset
from torchvision import transforms
from tqdm.auto import tqdm
from transformers import AutoTokenizer, PretrainedConfig
import time
import diffusers
from diffusers import (
    AutoencoderKL,
    DDPMScheduler,
    DiffusionPipeline,
    DPMSolverMultistepScheduler,
    DDIMScheduler,
    StableDiffusionPipeline,
    UNet2DConditionModel,
)
# https://github.com/huggingface/diffusers/blob/v0.22.0/src/diffusers/loaders.py
from diffusers.loaders import (
    LoraLoaderMixin,
    text_encoder_lora_state_dict,
)
from diffusers.models.attention_processor import (
    AttnAddedKVProcessor,
    AttnAddedKVProcessor2_0,
    SlicedAttnAddedKVProcessor,
)
from diffusers.models.lora import LoRALinearLayer
from diffusers.optimization import get_scheduler
from diffusers.training_utils import unet_lora_state_dict
from diffusers.utils import check_min_version, is_wandb_available
from diffusers.utils.import_utils import is_xformers_available

import wandb
import os.path as osp
from collections import defaultdict
from diffusers.utils.torch_utils import randn_tensor

def load_token_embedding(text_encoder, tokenizer, weight_path):
    # logger.info(f"Loading Token Embeddings from {weight_path}")
    # Load the saved token embeddings
    loaded_embeds_dict = torch.load(weight_path)
    # Get the input embedding layer
    token_embeddings = text_encoder.get_input_embeddings()
    # Process each token
    for token, embed in loaded_embeds_dict.items():
        # Check if token already exists in tokenizer
        token_id = tokenizer.convert_tokens_to_ids(token)
        if token_id == tokenizer.unk_token_id:
            print(f'adding {token} to the tokenizer vocabs')
            # Token doesn't exist, add to tokenizer
            tokenizer.add_tokens([token])
            token_id = tokenizer.convert_tokens_to_ids(token)
            # Resize the embedding layer to match new vocab size
            text_encoder.resize_token_embeddings(len(tokenizer))
        # Set the embedding weight
        with torch.no_grad():
            print(f'loading embedding for {token}')
            token_embeddings.weight[token_id] = embed.to(token_embeddings.weight.device)
            

In [75]:
# should have lora in the text encoder
pretrained_model_name_or_path = "stablediffusionapi/chilloutmix"

load_lora_weight_path = "data_root/logs/ch.ct.l4.kv_chiquita10-V_pr1.00_ln.lr1e-4.ti5e-4_b1g1/checkpoint-500"
load_token_embedding_path = "data_root/logs/ch.ct.l4.kv_chiquita10-V_pr1.00_ln.lr1e-4.ti5e-4_b1g1/checkpoint-500"
gen_dtype = torch.float16

pipeline = DiffusionPipeline.from_pretrained(pretrained_model_name_or_path, torch_dtype=gen_dtype)

if load_lora_weight_path is not None:
    # load attention processors
    print(25*"#")
    print('loading LoRA weight')
    pipeline.load_lora_weights(load_lora_weight_path, weight_name="pytorch_lora_weights.safetensors")
else: print('not loading loRA weight')
if load_token_embedding_path is not None:
    print('loading token embedding')
    load_token_embedding(pipeline.text_encoder, pipeline.tokenizer, osp.join(load_token_embedding_path,'token_embedding.pt'))
    


vae/diffusion_pytorch_model.safetensors not found
Loading pipeline components...: 100%|███████████████████████████████████████████| 7/7 [00:03<00:00,  1.97it/s]


#########################
loading LoRA weight
loading token embedding
adding v1 to the tokenizer vocabs


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


loading embedding for v1


In [76]:
pipeline.text_encoder

CLIPTextModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49409, 768)
      (position_embedding): Embedding(77, 768)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): PatchedLoraProjection(
              (regular_linear_layer): Linear(in_features=768, out_features=768, bias=True)
              (lora_linear_layer): LoRALinearLayer(
                (down): Linear(in_features=768, out_features=4, bias=False)
                (up): Linear(in_features=4, out_features=768, bias=False)
              )
            )
            (v_proj): PatchedLoraProjection(
              (regular_linear_layer): Linear(in_features=768, out_features=768, bias=True)
              (lora_linear_layer): LoRALinearLayer(
                (down): Linear(in_features=768, out_features=4, bias=False)
                (up): Linear(in_features=4, o

In [ ]:
# should NOT have lora in the text encoder
pretrained_model_name_or_path = "stablediffusionapi/chilloutmix"

load_lora_weight_path = "data_root/logs/ch.c.l4.kv_chiquita10-V_pr1.00_ln.lr1e-4.ti5e-4_b1g1/checkpoint-500"
load_token_embedding_path = "data_root/logs/ch.c.l4.kv_chiquita10-V_pr1.00_ln.lr1e-4.ti5e-4_b1g1/checkpoint-500"
gen_dtype = torch.float16

pipeline = DiffusionPipeline.from_pretrained(pretrained_model_name_or_path, torch_dtype=gen_dtype)

if load_lora_weight_path is not None:
    # load attention processors
    print(25*"#")
    print('loading LoRA weight')
    pipeline.load_lora_weights(load_lora_weight_path, weight_name="pytorch_lora_weights.safetensors")
else: print('not loading loRA weight')
if load_token_embedding_path is not None:
    print('loading token embedding')
    load_token_embedding(pipeline.text_encoder, pipeline.tokenizer, osp.join(load_token_embedding_path,'token_embedding.pt'))
    


/home/nessessence/anaconda3/envs/mace/lib/python3.10/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
vae/diffusion_pytorch_model.safetensors not found
Loading pipeline components...:  57%|████████████████████████▌                  | 4/7 [00:02<00:02,  1.37it/s]/home/nessessence/anaconda3/envs/mace/lib/python3.10/site-packages/transformers/models/clip/feature_extraction_clip.py:28: FutureWarning: The class CLIPFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use CLIPImageProcessor instead.
  warnings.warn(
Loading pipeline components...: 100%|███████████████████████████████████████████| 7/7 [00:03<00:00,  2.33it/s]


#########################
loading LoRA weight
loading token embedding
adding v1 to the tokenizer vocabs
loading embedding for v1


In [78]:
pipeline.text_encoder


CLIPTextModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49409, 768)
      (position_embedding): Embedding(77, 768)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e

In [ ]:


# concept = 'sceleb5g0' # moodeng
concept = 'gout' # moodeng
data_setting = 'small' # full
is_relearn = False # True

pretrained = 'ch' # "rv" # sd1.4
batch_size = 1
gradient_accumulation_steps = 4 #4


lora_rank = 4 # 1
lora_alpha = None  # None

use_te = True
use_pr = True
use_nis = [False]
use_ti = True # True 

seeds = [0]  # List of seeds

lr_scheduler = 'linear' # 'linear' # 'cosine' # 'constant'
# lr_scheduler = 'constant' # 'linear' # 'cosine' # 'constant'
lr_lora_grid = ["1e-4"]
# lr_ti_grid   = ["5e-2"]   # only used if use_ti
lr_ti_grid   = ["5e-4"]   # only used if use_ti


lr_lora_te_grid = ["1e-5"]
# Create all combinations of lr_lora, lr_ti, and seed
combos = list(itertools.product(lr_lora_grid,
                                lr_ti_grid if use_ti else [None],
                                lr_lora_te_grid if use_te else [None],
                                seeds,use_nis ))
chunk_id = 0
total_chunks = 1
combos = get_chunk(combos, chunk_id, total_chunks=total_chunks)

final_exp_names = []
for lr_lora, lr_ti, lr_te, seed, use_ni in combos:

    if data_setting == 'fewshot':
        
        if 'sceleb' in concept:
            dataset_name = f'{concept}U3'
        elif concept == 'avp':
            dataset_name = 'avpS3'
        else:
            dataset_name = f'{concept}U3'
    if data_setting == 'small':
        
        if 'sceleb' in concept:
            dataset_name = f'{concept}N10'
        else:
            dataset_name = f'{concept}10'
    else:
        
        if 'sceleb' in concept:
            dataset_name = f'{concept}N50'
        elif concept == 'avp':
            dataset_name = 'avp20'
        else:
            dataset_name = f'{concept}50'


    # however, if use_ti is True, the prompt will be changed to 'A photo of a v1' for all concepts
    data_root = dataset_name2data_root[dataset_name]
    if use_ti:
        if 'sceleb' in concept:
            prompt = 'A photo of a v1,A photo of a v2,A photo of a v3,A photo of a v4,A photo of a v5'
            placeholder_token = 'v1,v2,v3,v4,v5'
        else:
            # prompt = 'A photo of a v1' 
            prompt = 'a photo of v1' 
            placeholder_token = 'v1'
    else:
        if concept in concept2prompt:
            prompt = concept2prompt[concept]
        else: 
            # prompt = 'sks person'    
            prompt = 'A photo of sks person'    
            from diffusers import DiffusionPipeline

# pipe = DiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5")

    if pretrained == 'sd1.4':
        pretrained_path = 'CompVis/stable-diffusion-v1-4' 
    elif pretrained == 'sd1.5':
        pretrained_path = 'runwayml/stable-diffusion-v1-5'
    if pretrained == 'rv':
        pretrained_path = 'stablediffusionapi/realistic-vision-v51'
    if pretrained == 'ch':
        pretrained_path = 'stablediffusionapi/chilloutmix'
    if  is_relearn:
        pretrained_path = f"data_root/logs/erase_l1.{concept}VPr.object_lr2.5e-4/LoRA_fusion_model"

            
    if use_ti:
        dataset_name_for_exp = dataset_name + "-V"
        if use_ni:
            dataset_name_for_exp += ".r"
    else: dataset_name_for_exp = dataset_name

    if use_te:
        exp_name = f'ct.l{lora_rank}.kv'
    else: exp_name = f'c.l{lora_rank}.kv'
    
    if lora_alpha:
        exp_name += f'.a{lora_alpha}'
    
    exp_name += f'_{dataset_name_for_exp}'
    
    
    if pretrained == 'sd1.5':
        exp_name = f'sd15.{exp_name}'
    if pretrained == 'sd1.4':
        exp_name = f'sd14.{exp_name}' 
    elif pretrained == 'rv':
        exp_name = f'rv.{exp_name}'
    elif pretrained == 'ch':
        exp_name = f'ch.{exp_name}'
    
    if use_pr:
        exp_name += f'_pr1.00'
        
    if lr_scheduler == 'constant':
        exp_name += '_lr'
    elif lr_scheduler == 'linear':
        exp_name += '_ln.lr'
        
        
    if lora_rank >0: exp_name += f"{str(lr_lora)}"
    if use_ti:
        exp_name += f'.ti{str(lr_ti)}'
    exp_name += f'_b{batch_size}g{gradient_accumulation_steps}'
    if is_relearn:
        unlearn_setting = pretrained_path.split("/")[-2].split("_")[1]
        exp_name = f'uul.{unlearn_setting}_{exp_name}'
        
    if use_ni: initializer_token = ''
    else: 
        initializer_token = concept2initializer[concept]

    prior_folder = 'original_pretrained'
    if pretrained == 'sd1.5':
        prior_folder = 'original_pretrained_sd1.5'
    if pretrained == 'sd1.4':
        prior_folder = 'original_pretrained_sd1.4'     
    if pretrained == 'rv':
        prior_folder = 'original_realistic_vision'
    elif pretrained == 'ch':
        prior_folder = 'original_chilloutmix'

    name_tag = ''
    if is_relearn: name_tag += 'uul'
    name_tag = f'{name_tag} {dataset_name}'
    name_tag += f' l{lora_rank}'
    if use_ti: 
        # name_tag += f' ti.{lr_ti}'
        name_tag += f' ti'


    max_train_steps = 2000
    if data_setting == 'fewshot':
        max_train_steps = 1000

    if 'sceleb' in concept:
        if data_setting == 'fewshot' or data_setting == 'small' :
            max_train_steps = 3000
        else:
            max_train_steps = 50000
    if lr_scheduler == 'linear' and data_setting == 'small':
        max_train_steps = 1000
    if lr_scheduler == 'linear' and data_setting == 'full':
        max_train_steps = 2000        
    if seed != 0:
        exp_name += f'.r{seed}'
        name_tag += f' r{seed}'
    
    script = f"""
    accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path={pretrained_path}  \\
    --instance_data_dir={data_root} \\
    --output_dir="data_root/logs/{exp_name}" \\
    --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
    --train_batch_size={batch_size} --gradient_accumulation_steps={gradient_accumulation_steps} \\
    --lora_rank {lora_rank} --target_lora_modules to_k to_v --target_lora_layers cross \\
    --max_train_steps={max_train_steps}  --validation_steps=50  --checkpointing_steps=50 --seed {seed} \\
    --lr_scheduler "{lr_scheduler}" \\
    --run_note '{name_tag}' \\"""
        
        
    if use_pr:

        script += f"""
    --with_prior_preservation --prior_loss_weight=1.0 --num_class_images 200 \\
    --class_prompt="a photo of a person" --class_data_dir="data_root/generated/model/{prior_folder}/a photo of a person/6.00" \\"""    


    if lora_alpha:
        script+= f"""
    --lora_alpha {lora_alpha} \\"""
    
    # if pretrained == 'rv':
    script += f"""
    --cfg_scale 6.0 \\"""
    

    # Conditional learning rate + TI options
    if use_ti:
        if lora_rank <= 0:
            script += f"""
    --learning_rate_ti {lr_ti} \\
    --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
        else:
            if use_te:
                script += f"""
    --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
    --train_text_encoder --learning_rate_lora_text_encoder {lr_te} \\
    --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
            else:
                script += f"""
    --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
    --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
    else:
        script += f"""
    --learning_rate {lr_lora}"""



    print(script)
    # print(exp_name)

    final_exp_names += [exp_name]
print(final_exp_names)